# Умный светофор: детекция машин, оценка загруженности и адаптивное управление сигналом**Capstone-проект по курсу AI/ML Fundamentals — Individual Project Track****Автор:** _[ВПИШИ СВОЁ ПОЛНОЕ ИМЯ]_## 1. Постановка проблемы**Проблема:** на нерегулируемых по трафику светофорах зелёный сигнал горит фиксированное время независимо от реальной загруженности перекрёстка. Это приводит к лишним заторам в час пик и впустую потраченному времени в периоды низкого трафика.**Стейкхолдер:** городские службы управления дорожным движением / водители, стоящие в очереди.**ML-задача:** Object Detection (детекция объектов) + Multi-Object Tracking — находим и отслеживаем автомобили на видео с камеры перекрёстка. Input: кадр видео. Output: bounding boxes машин + ID трека + производные метрики (число машин в очереди, время ожидания).**Критерий успеха:** модель находит машины с mAP@50 не ниже базового порога (сравниваем с baseline), пайплайн считает очередь и на основе порога принимает решение об изменении длительности зелёного сигнала.**Scope проекта:** прототип на записанном видео (не real-time продакшн-система). Демонстрируется полный пайплайн: детекция -> трекинг -> анализ очереди -> решение по сигналу.## 2. Пайплайн проекта1. **Baseline** — детекция машин без ML (background subtraction, классическое CV)2. **Основная модель** — YOLOv8, дообученная (fine-tuned) на датасете машин3. **Сравнение подходов** — baseline vs YOLO pretrained vs YOLO fine-tuned4. **Трекинг** — ByteTrack для отслеживания машин между кадрами5. **Анализ очереди** — подсчёт машин и времени ожидания6. **Логика решения** — пороговое правило управления зелёным сигналом7. **Оценка на unseen data + error analysis**8. **Responsible AI** — ограничения, риски, честность модели

## Шаг 0. Установка библиотек

In [ ]:
!pip install -q ultralytics yt-dlp opencv-python-headless roboflow pandas matplotlib scikit-learnimport cv2import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom ultralytics import YOLOimport os, json, timeprint("Всё установлено")

## Шаг 1. Данные**Источник:** [ВПИШИ: Roboflow Universe dataset name / ссылка] — набор размеченных изображений машин на перекрёстках, формат YOLOv8 (bounding boxes + класс).**Почему подходит:** изображения сняты с камер, похожих по ракурсу на светофорные, что важно для generalization на наше тестовое видео.Дообучение (fine-tuning) вместо обучения с нуля: используем веса YOLOv8, предобученные на COCO (80 классов, включая машины), и дообучаем только под наш специфичный ракурс — это требует значительно меньше данных и времени.

In [ ]:
from roboflow import Roboflowrf = Roboflow(api_key="ВСТАВЬ_СВОЙ_КЛЮЧ_ЗДЕСЬ")  # roboflow.com -> Settings -> API Key# Датасет: Traffic Intersection Vehicle Detection (VAI), 4526 изображений,# классы car/truck/bus/motorbike/person, лицензия CC BY 4.0# https://universe.roboflow.com/vai/traffic-intersection-vehicle-detectionproject = rf.workspace("vai").project("traffic-intersection-vehicle-detection")# ВАЖНО: зайди на страницу датасета -> вкладка "Dataset" -> посмотри номер версии# (их там несколько, отличаются количеством изображений). Начни с version(1) -# если хочешь именно полные 4526 изображений, проверь на сайте какая версия самая полная.dataset = project.version(1).download("yolov8")print("Датасет скачан в:", dataset.location)

## Шаг 2. EDA — краткий анализ датасетаСмотрим на размер выборки, баланс классов и распределение размеров объектов — это нужно для критерия "EDA and issue identification".

In [ ]:
import yaml, globwith open(dataset.location + '/data.yaml') as f:    data_cfg = yaml.safe_load(f)print("Классы в датасете:", data_cfg['names'])train_images = glob.glob(dataset.location + '/train/images/*')val_images = glob.glob(dataset.location + '/valid/images/*')test_images = glob.glob(dataset.location + '/test/images/*') if os.path.exists(dataset.location + '/test') else []print(f"Train: {len(train_images)} изображений")print(f"Val:   {len(val_images)} изображений")print(f"Test:  {len(test_images)} изображений")# Подсчёт количества объектов на класс по train labelsfrom collections import Counterclass_counts = Counter()label_files = glob.glob(dataset.location + '/train/labels/*.txt')for lf in label_files:    with open(lf) as f:        for line in f:            cls_id = int(line.split()[0])            class_counts[data_cfg['names'][cls_id]] += 1print("\nБаланс классов (train):")for cls, cnt in class_counts.items():    print(f"  {cls}: {cnt}")plt.figure(figsize=(8,4))plt.bar(class_counts.keys(), class_counts.values())plt.title('Распределение классов в train-выборке')plt.ylabel('Количество объектов')plt.xticks(rotation=45)plt.tight_layout()plt.savefig('eda_class_balance.png')plt.show()

**Наблюдения по EDA:** _[впиши свои наблюдения после запуска — например: "класс 'car' доминирует над 'truck'/'bus', что ожидаемо для городского трафика; возможен слабый bias модели в сторону легковых машин"]_## Шаг 2b. Data Gate: проверка дубликатов и утечки между сплитами**Это обязательная проверка перед моделированием (M8C3 Data Gate).** Главный риск для CV-датасетов: кадры из одного и того же видео/сцены могут случайно попасть и в train, и в test — тогда модель будет "запоминать" сцену, а метрики на test будут завышены (data leakage).Проверяем точные дубликаты через хэш файла (MD5) между train/valid/test.

In [ ]:
import hashlibdef hash_file(path):    with open(path, 'rb') as f:        return hashlib.md5(f.read()).hexdigest()def collect_hashes(folder):    hashes = {}    for img_path in glob.glob(folder + '/images/*'):        hashes[hash_file(img_path)] = img_path    return hashestrain_hashes = collect_hashes(dataset.location + '/train')valid_hashes = collect_hashes(dataset.location + '/valid')test_hashes = collect_hashes(dataset.location + '/test') if test_images else {}train_valid_overlap = set(train_hashes) & set(valid_hashes)train_test_overlap = set(train_hashes) & set(test_hashes)valid_test_overlap = set(valid_hashes) & set(test_hashes)overlap_report = pd.DataFrame([    {'split_pair': 'train-valid', 'exact_duplicate_images': len(train_valid_overlap)},    {'split_pair': 'train-test', 'exact_duplicate_images': len(train_test_overlap)},    {'split_pair': 'valid-test', 'exact_duplicate_images': len(valid_test_overlap)},])print(overlap_report)overlap_report.to_csv('duplicate_and_group_check.csv', index=False)if len(train_test_overlap) > 0 or len(train_valid_overlap) > 0:    print("\nВНИМАНИЕ: найдены точные дубликаты между сплитами — это утечка данных (data leakage).")    print("Задокументируй это в docs/data_audit.md как DQ-01 и укажи, как это влияет на интерпретацию метрик.")else:    print("\nТочных дубликатов между сплитами не найдено (по MD5-хэшу файла).")    print("Примечание: это не исключает near-duplicate кадров (например, соседние кадры одного видео,")    print("отличающиеся на пиксель) — если датасет явно собран из видео, стоит визуально проверить")    print("несколько случайных пар изображений из train и test на предмет визуального сходства сцены.")

**Результат проверки (заполни после запуска):** _[впиши: найдены ли точные дубликаты; если да — сколько и в каких сплитах; как это повлияло на интерпретацию финальных метрик в Шаге 7]_## Шаг 3. Baseline-модель (без машинного обучения)Для честного сравнения нужен baseline — классический подход без нейросети. Используем **background subtraction** (MOG2): вычисляем "фоновое" изображение дороги без машин, и всё, что отличается от фона в кадре, считаем потенциальным объектом (движущейся машиной).Это простой, интерпретируемый, но менее точный метод — хорошая база для сравнения с YOLO.

In [ ]:
def baseline_detect_video(video_path, max_frames=200):    """Baseline детекция машин через background subtraction (без ML)."""    cap = cv2.VideoCapture(video_path)    back_sub = cv2.createBackgroundSubtractorMOG2(history=200, varThreshold=40, detectShadows=True)    counts_per_frame = []    frame_idx = 0    while cap.isOpened() and frame_idx < max_frames:        ret, frame = cap.read()        if not ret:            break        fg_mask = back_sub.apply(frame)        _, fg_mask = cv2.threshold(fg_mask, 250, 255, cv2.THRESH_BINARY)        contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)        # Фильтруем мелкий шум по площади контура        car_like_contours = [c for c in contours if cv2.contourArea(c) > 500]        counts_per_frame.append(len(car_like_contours))        frame_idx += 1    cap.release()    return counts_per_frame# Baseline посчитаем на тестовом видео уже после того как скачаем его (Шаг 6)print("Функция baseline готова")

**Ограничение baseline:** background subtraction плохо работает при остановившихся надолго машинах (они "сливаются" с фоном) и чувствителен к теням/освещению — это ожидаемо хуже, чем YOLO, и именно поэтому нам нужна ML-модель.## Шаг 4. Основная модель — дообучение YOLOv8 (Experiment tracking)Ключевые понятия:- **Epoch** — один проход по всем данным- **Loss** — мера ошибки модели (box_loss, cls_loss, dfl_loss для YOLO)- **mAP@50** — главная метрика детекции (точность+полнота при IoU 0.5)- **Transfer learning** — стартуем с весов, обученных на COCO, а не со случайныхНиже — таблица экспериментов с разными гиперпараметрами (для критерия "Experiments and tuning" и "Experiment tracking").

In [ ]:
experiment_log = []def run_experiment(name, epochs, imgsz, lr0=0.01, batch=16):    model = YOLO('yolov8n.pt')    start = time.time()    results = model.train(        data=dataset.location + '/data.yaml',        epochs=epochs,        imgsz=imgsz,        batch=batch,        lr0=lr0,        patience=5,        project='traffic_light_project',        name=name,        verbose=False    )    duration = time.time() - start    metrics = model.val()    experiment_log.append({        'experiment': name,        'epochs': epochs,        'imgsz': imgsz,        'lr0': lr0,        'batch': batch,        'mAP50': float(metrics.box.map50),        'mAP50-95': float(metrics.box.map),        'precision': float(metrics.box.mp),        'recall': float(metrics.box.mr),        'train_time_sec': round(duration, 1)    })    return model# Эксперимент 1: базовые настройки, немного эпох (быстрый прогон)model_exp1 = run_experiment('exp1_baseline_settings', epochs=10, imgsz=416)# Эксперимент 2: больше эпох, стандартный размер изображения (основная модель)model_exp2 = run_experiment('exp2_more_epochs', epochs=25, imgsz=640)# Эксперимент 3: другой learning rate (проверяем чувствительность)model_exp3 = run_experiment('exp3_lower_lr', epochs=25, imgsz=640, lr0=0.001)exp_df = pd.DataFrame(experiment_log)print(exp_df)exp_df.to_csv('experiment_log.csv', index=False)

**Выбор финальной модели:** _[после запуска впиши, например: "exp2 показал лучший баланс mAP50 и времени обучения; exp3 с меньшим lr не дал прироста качества, что говорит о том, что дефолтный lr0=0.01 уже близок к оптимальному для этой задачи"]_Финальную модель сохраняем отдельно как `best_model.pt` — она пригодится для inference-демо (Шаг 8), чтобы демо не зависело от переменных в памяти ноутбука.

In [ ]:
BEST_EXPERIMENT = 'exp2_more_epochs'  # поменяй, если выбрала другой эксперимент по итогам таблицы вышеbest_weights_path = f'traffic_light_project/{BEST_EXPERIMENT}/weights/best.pt'final_model = YOLO(best_weights_path)# Сохраняем финальные веса отдельно — это и есть "saved model artifact" по критерию 5import shutilos.makedirs('artifacts', exist_ok=True)shutil.copy(best_weights_path, 'artifacts/best_model.pt')print("Финальная модель сохранена в artifacts/best_model.pt")

## Шаг 5. Графики обучения (loss и метрики) для выбранной модели

In [ ]:
results_csv = f'traffic_light_project/{BEST_EXPERIMENT}/results.csv'df = pd.read_csv(results_csv)df.columns = df.columns.str.strip()fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(df['epoch'], df['train/box_loss'], label='train box_loss')axes[0].plot(df['epoch'], df['val/box_loss'], label='val box_loss')axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('Loss')axes[0].set_title('Функция потерь по эпохам'); axes[0].legend()axes[1].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP@50', color='green')axes[1].set_xlabel('Эпоха'); axes[1].set_ylabel('mAP')axes[1].set_title('mAP@50 по эпохам'); axes[1].legend()plt.tight_layout()plt.savefig('training_curves.png')plt.show()

## Шаг 6. Тестовое видео (unseen data для итоговой оценки)Важно: это видео используется только для демо и качественной оценки трекинга/логики; количественная оценка детекции (mAP) идёт по test-split датасета из Roboflow (Шаг 7), который модель не видела при обучении.

In [ ]:
YOUTUBE_URL = "ССЫЛКА_НА_ВИДЕО_С_ПЕРЕКРЁСТКОМ"  # замени на реальную ссылку!yt-dlp -f 'best[height<=480]' -o 'traffic_video.mp4' {YOUTUBE_URL}print("Видео скачано")

## Шаг 7. Оценка на unseen data + сравнение с baselineСравниваем три подхода:1. Baseline (background subtraction, без ML)2. YOLOv8 pretrained (без дообучения — "из коробки")3. YOLOv8 fine-tuned (наша основная модель)на **test-split** датасета (данные, не участвовавшие ни в обучении, ни в валидации).

In [ ]:
# 1. Метрики fine-tuned модели на test-split (unseen data)test_metrics_finetuned = final_model.val(data=dataset.location + '/data.yaml', split='test')print("Fine-tuned YOLOv8 — mAP50:", float(test_metrics_finetuned.box.map50))# 2. Метрики pretrained модели (без дообучения) на том же test-splitpretrained_model = YOLO('yolov8n.pt')test_metrics_pretrained = pretrained_model.val(data=dataset.location + '/data.yaml', split='test')print("Pretrained YOLOv8 (без fine-tuning) — mAP50:", float(test_metrics_pretrained.box.map50))# 3. Baseline — считаем на нескольких тестовых изображениях количество найденных контуров# Baseline работает на видео, а не на статичных изображениях по одному кадру, поэтому# для честности сравнения оцениваем его отдельно на видео ниже (Шаг 8), а количественно# сравниваем только детекторы, применимые к статичным изображениям (pretrained vs fine-tuned).comparison = pd.DataFrame([    {'model': 'YOLOv8 pretrained (no fine-tuning)', 'mAP50': float(test_metrics_pretrained.box.map50),     'precision': float(test_metrics_pretrained.box.mp), 'recall': float(test_metrics_pretrained.box.mr)},    {'model': 'YOLOv8 fine-tuned (наша модель)', 'mAP50': float(test_metrics_finetuned.box.map50),     'precision': float(test_metrics_finetuned.box.mp), 'recall': float(test_metrics_finetuned.box.mr)},])print(comparison)comparison.to_csv('model_comparison.csv', index=False)

## Шаг 8. Error AnalysisСмотрим на конкретные случаи, где модель ошибается — это обязательный пункт критерия 4.

In [ ]:
# Прогоняем финальную модель на нескольких test-изображениях и визуально ищем ошибкиtest_imgs = glob.glob(dataset.location + '/test/images/*')[:8]fig, axes = plt.subplots(2, 4, figsize=(18, 8))for ax, img_path in zip(axes.flatten(), test_imgs):    result = final_model.predict(img_path, conf=0.25, verbose=False)[0]    annotated = result.plot()    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))    ax.set_title(os.path.basename(img_path), fontsize=8)    ax.axis('off')plt.tight_layout()plt.savefig('error_analysis_samples.png')plt.show()print("Просмотри картинки выше и найди минимум 2-3 характерные ошибки:")print("- Пропущенные машины (false negatives) — например, частично перекрытые другими машинами")print("- Ложные срабатывания (false positives) — например, тень или столб распознан как машина")print("- Путаница классов — например, автобус распознан как грузовик")

**Разбор ошибок (заполни после просмотра картинок выше):**- _[Пример] Модель пропускает частично перекрытые машины в плотном потоке (occlusion) — это ожидаемая слабость детекторов на одном кадре без учёта контекста между кадрами._- _[Пример] Ложные срабатывания на тенях от деревьев при ярком солнце — можно улучшить фильтрацией по confidence threshold или добавлением таких кейсов в обучающую выборку._- _[Впиши свой третий кейс]_**Robustness / edge cases:** _[впиши, например: "проверила модель на кадре с дождём/ночным освещением — качество детекции снижается, что документируем в разделе Limitations"]_## Шаг 9. Детекция + трекинг на видео (демо-пайплайн)Используем ByteTrack для отслеживания машин между кадрами — считаем время ожидания каждой машины.

In [ ]:
def process_traffic_video(video_path, model, output_path='output_annotated.mp4'):    cap = cv2.VideoCapture(video_path)    fps = cap.get(cv2.CAP_PROP_FPS)    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))    stationary_frames = {}    prev_positions = {}    MOVEMENT_THRESHOLD = 5    history = []    frame_idx = 0    while cap.isOpened():        ret, frame = cap.read()        if not ret:            break        results = model.track(frame, persist=True, classes=[2, 3, 5, 7], verbose=False)        waiting_count, total_count = 0, 0        if results[0].boxes.id is not None:            boxes = results[0].boxes.xywh.cpu().numpy()            track_ids = results[0].boxes.id.cpu().numpy().astype(int)            total_count = len(track_ids)            for box, tid in zip(boxes, track_ids):                cx, cy = box[0], box[1]                if tid in prev_positions:                    px, py = prev_positions[tid]                    dist = np.hypot(cx - px, cy - py)                    stationary_frames[tid] = stationary_frames.get(tid, 0) + 1 if dist < MOVEMENT_THRESHOLD else 0                prev_positions[tid] = (cx, cy)                wait_seconds = stationary_frames.get(tid, 0) / fps                if wait_seconds > 2:                    waiting_count += 1        annotated = results[0].plot()        cv2.putText(annotated, f'Всего машин: {total_count}', (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 2)        cv2.putText(annotated, f'В очереди: {waiting_count}', (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)        out.write(annotated)        history.append({'frame': frame_idx, 'total': total_count, 'waiting': waiting_count})        frame_idx += 1    cap.release(); out.release()    return pd.DataFrame(history), fpshistory_df, fps = process_traffic_video('traffic_video.mp4', final_model)print("Готово, output_annotated.mp4 сохранён")

## Шаг 10. Логика управления светофором (rule-based decision)Простое интерпретируемое правило поверх результатов детекции/трекинга. Явно не отдельная нейросеть — обоснование см. в README (Limitations).

In [ ]:
def decide_signal(waiting_count, base_green=30, extend_step=5, max_green=60, min_green=15):    if waiting_count >= 8:        return min(base_green + extend_step*2, max_green), "Большая очередь -> продлеваем зелёный"    elif waiting_count >= 4:        return min(base_green + extend_step, max_green), "Средняя очередь -> немного продлеваем"    elif waiting_count <= 1:        return max(base_green - extend_step, min_green), "Почти нет машин -> сокращаем зелёный"    else:        return base_green, "Норма -> без изменений"recent_waiting = history_df['waiting'].tail(int(fps * 5)).mean()green_time, decision = decide_signal(round(recent_waiting))print(f"Среднее число машин в очереди (последние 5 сек): {recent_waiting:.1f}")print(f"Решение: {decision}")print(f"Новая длительность зелёного: {green_time} сек")plt.figure(figsize=(12,4))plt.plot(history_df['frame']/fps, history_df['total'], label='Всего машин')plt.plot(history_df['frame']/fps, history_df['waiting'], label='В очереди')plt.xlabel('Время (сек)'); plt.ylabel('Количество машин')plt.title('Загруженность перекрёстка во времени'); plt.legend()plt.savefig('traffic_over_time.png')plt.show()

## Шаг 11. Input validation (обработка некорректного входа)По критерию 5 нужна обработка невалидных входов — проверяем формат/существование видео перед обработкой.

In [ ]:
def safe_process_video(video_path, model):    if not os.path.exists(video_path):        raise FileNotFoundError(f"Файл не найден: {video_path}")    cap = cv2.VideoCapture(video_path)    if not cap.isOpened():        raise ValueError(f"Не удалось открыть видео (повреждён или неверный формат): {video_path}")    ret, _ = cap.read()    if not ret:        raise ValueError("Видео пустое или повреждено — не удалось прочитать первый кадр")    cap.release()    print("Видео прошло проверку, можно обрабатывать")    return True# Пример: тест на несуществующем файле (должен вернуть понятную ошибку, а не упасть с трассировкой)try:    safe_process_video('несуществующий_файл.mp4', final_model)except (FileNotFoundError, ValueError) as e:    print(f"Обработано корректно: {e}")# Проверка реального видеоsafe_process_video('traffic_video.mp4', final_model)

## Шаг 12. Полностью воспроизводимое inference-демо (с нуля, из сохранённых артефактов)Это ключевая часть для критерия "Clean-runtime reproducibility" — загружаем модель **заново из файла**, а не используем переменную `final_model` из памяти, чтобы показать, что демо реально работает независимо от процесса обучения выше.

In [ ]:
# Имитация "чистого" запуска: загружаем модель только из сохранённого файлаinference_model = YOLO('artifacts/best_model.pt')demo_history, demo_fps = process_traffic_video('traffic_video.mp4', inference_model, output_path='demo_output.mp4')demo_waiting = demo_history['waiting'].tail(int(demo_fps*5)).mean()demo_green, demo_decision = decide_signal(round(demo_waiting))print("=== ИТОГОВОЕ ДЕМО ===")print(f"Машин в очереди: {round(demo_waiting)}")print(f"Решение по светофору: {demo_decision}")print(f"Новое время зелёного: {demo_green} сек")print("Аннотированное видео: demo_output.mp4")

## 13. Responsible AI и ограничения**Bias и репрезентативность:**- Датасет содержит преимущественно [впиши: дневное время / определённый тип перекрёстков / определённую страну съёмки] — модель может хуже работать в других условиях (ночь, снег, другие типы дорог).- Класс "car" численно доминирует над "bus"/"truck" в обучающей выборке (см. EDA, Шаг 2) — модель может быть менее точна на редких классах транспорта.**Приватность и безопасность:**- Видео с камер трафика потенциально содержит номерные знаки и лица водителей/пешеходов — в реальном развёртывании требуется анонимизация (блюр номеров/лиц) перед хранением или обработкой.- Система принимает решения, влияющие на реальный трафик — ошибки детекции могут привести к неоптимальному или небезопасному управлению сигналом, поэтому prototype не пригоден для прямого управления реальным светофором без дополнительного тестирования и предохранительных механизмов (fail-safe defaults).**Ограничения:**- Оценка "длины очереди" в метрах не выполняется — используется количество машин как прокси (нет калибровки камеры).- Логика управления сигналом — простое пороговое правило, не оптимизационный алгоритм (RL или подобное) — обосновано ограничением по времени проекта.- Модель обучена на ограниченном датасете и числе эпох (см. Experiment Log) — для продакшена нужно больше данных и разнообразия условий съёмки.- Прототип не учитывает синхронизацию соседних перекрёстков ("зелёная волна") и несколько полос/направлений отдельно.**Уместное использование:** прототип предназначен для демонстрации подхода и не должен использоваться для управления реальной инфраструктурой без дополнительной валидации, тестирования безопасности и одобрения регулирующих органов.## 14. Выводы_[Впиши после всех запусков, 3-5 предложений: что получилось, какая модель финально выбрана и почему, как соотносится с baseline, какие метрики достигнуты, что стоит улучшить в первую очередь]_